# Fine-tuning do MedGemma com QLoRA
Aplica QLoRA no MedGemma 4B usando o dataset ISIC 2020.

In [ ]:
!pip install -q transformers accelerate peft trl bitsandbytes huggingface_hub

In [ ]:
import sys
sys.path.append('/kaggle/working/melanoma-tcc')

import pandas as pd
from kaggle_secrets import UserSecretsClient
from src.data.preprocessing import ISICDataset, split_dataframe
from src.model.finetuning import load_model_for_finetuning, apply_lora, get_trainer

secrets = UserSecretsClient()
HF_TOKEN = secrets.get_secret('HF_TOKEN')

CSV_PATH = '/kaggle/input/siim-isic-melanoma-classification/train.csv'
IMAGES_DIR = '/kaggle/input/siim-isic-melanoma-classification/train'
OUTPUT_DIR = '/kaggle/working/medgemma-melanoma'

In [ ]:
model, processor = load_model_for_finetuning(HF_TOKEN)
model = apply_lora(model)

In [ ]:
df = pd.read_csv(CSV_PATH)
# Amostra balanceada para fine-tuning inicial
pos = df[df['target'] == 1].sample(500, random_state=42)
neg = df[df['target'] == 0].sample(500, random_state=42)
balanced_df = pd.concat([pos, neg]).reset_index(drop=True)

train_df, val_df = split_dataframe(balanced_df, val_ratio=0.1)
train_df.to_csv('/kaggle/working/train.csv', index=False)
val_df.to_csv('/kaggle/working/val.csv', index=False)

train_dataset = ISICDataset('/kaggle/working/train.csv', IMAGES_DIR, processor, split='train')
val_dataset = ISICDataset('/kaggle/working/val.csv', IMAGES_DIR, processor, split='val')

print(f'Train: {len(train_dataset)} | Val: {len(val_dataset)}')

In [ ]:
trainer = get_trainer(model, processor, train_dataset, val_dataset, OUTPUT_DIR)
trainer.train()
trainer.save_model(OUTPUT_DIR)
print(f'Modelo salvo em {OUTPUT_DIR}')